### Refund Agent — Evaluation

Standalone evaluation task for the deployed Refund Agent. Runs after
`Refund_Recommender_Agent` in the pipeline so the endpoint is up before
we hit it.

- **Gated by `SKIP_EVAL`** (default `"true"`) — flip to `"false"` to actually
  run the evaluation. The default skip is a deliberate rate-limit safeguard:
  evaluation is the burst-iest LLM consumer in the bundle and can hit the
  shared FMAPI endpoint's QPS cap when refund + complaint eval run
  back-to-back.
- Calls the deployed endpoint via `mlflow.deployments.get_deploy_client`
  rather than importing the agent module — so this notebook is fully
  self-contained and tests the actual production endpoint shape.
- Eval results land in `/Shared/{CATALOG}_refund_agent_dev`.

In [ ]:
%pip install -U -qqqq mlflow-skinny[databricks] databricks-sdk
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
ENDPOINT_NAME = dbutils.widgets.get("REFUND_AGENT_ENDPOINT_NAME")

try:
    SKIP_EVAL = dbutils.widgets.get("SKIP_EVAL").strip().lower() == "true"
except Exception:
    SKIP_EVAL = True

import mlflow

DEV_EXPERIMENT = f"/Shared/{CATALOG}_refund_agent_dev"
mlflow.set_experiment(DEV_EXPERIMENT)

print(f"Catalog:          {CATALOG}")
print(f"Endpoint:         {ENDPOINT_NAME}")
print(f"SKIP_EVAL:        {SKIP_EVAL}")
print(f"Dev experiment:   {DEV_EXPERIMENT}")
print(f"MLflow version:   {mlflow.__version__}")

In [ ]:
if SKIP_EVAL:
    print(
        f"⏭  SKIP_EVAL=true — skipping mlflow.genai.evaluate to avoid the ~20 "
        f"LM-call eval burst.  Pass --params \"SKIP_EVAL=false\" to actually run "
        f"the evaluation (e.g. before a demo or after a prompt change)."
    )
    dbutils.notebook.exit("skipped")

In [ ]:
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

POLL_INTERVAL_S = 15
MAX_POLLS = 40

print(f"Polling endpoint readiness ({MAX_POLLS * POLL_INTERVAL_S // 60} min max)…")

for attempt in range(1, MAX_POLLS + 1):
    try:
        ep = w.serving_endpoints.get(ENDPOINT_NAME)
        ready = str(getattr(ep.state, "ready", "")).upper() if ep.state else ""
        cfg_update = str(getattr(ep.state, "config_update", "")).upper() if ep.state else ""
        if "READY" in ready:
            print(f"  ✅ Endpoint READY (config_update={cfg_update or 'n/a'})")
            break
        print(f"  ⏳ [{attempt}/{MAX_POLLS}] ready={ready}, config_update={cfg_update}")
    except Exception as e:
        print(f"  ⚠️  poll error: {type(e).__name__}: {e}")
    time.sleep(POLL_INTERVAL_S)
else:
    raise RuntimeError(
        f"Endpoint {ENDPOINT_NAME} did not become READY within "
        f"{MAX_POLLS * POLL_INTERVAL_S // 60} minutes."
    )

In [ ]:
import os
import random
from mlflow.deployments import get_deploy_client

os.environ["MLFLOW_GENAI_EVAL_MAX_WORKERS"] = "1"
os.environ["MLFLOW_GENAI_EVAL_MAX_SCORER_WORKERS"] = "1"
# Bump client-side HTTP timeouts. Defaults (120s) trip up agent endpoints and
# the LLM judge calls inside scorers (Safety / RelevanceToQuery / Guidelines).
os.environ.setdefault("MLFLOW_DEPLOYMENT_PREDICT_TIMEOUT", "600")
os.environ.setdefault("MLFLOW_DEPLOYMENT_PREDICT_TOTAL_TIMEOUT", "900")
os.environ.setdefault("MLFLOW_HTTP_REQUEST_TIMEOUT", "600")

_deploy_client = get_deploy_client("databricks")


def _is_rate_limit(exc: BaseException) -> bool:
    msg = str(exc)
    return (
        "REQUEST_LIMIT_EXCEEDED" in msg
        or "RateLimitError" in type(exc).__name__
        or "rate limit" in msg.lower()
    )


def predict_fn(messages):
    """Call the deployed Refund Agent endpoint with retry-on-rate-limit."""
    max_attempts = 6
    for attempt in range(max_attempts):
        try:
            return _deploy_client.predict(
                endpoint=ENDPOINT_NAME,
                inputs={"messages": messages},
            )
        except Exception as exc:
            if not _is_rate_limit(exc) or attempt == max_attempts - 1:
                raise
            backoff = min(60, 2 ** attempt) + random.uniform(0, 1)
            print(f"  ⚠️  rate-limited (attempt {attempt + 1}/{max_attempts}), sleeping {backoff:.1f}s")
            time.sleep(backoff)


print(f"✅ predict_fn defined — will call endpoint {ENDPOINT_NAME}")

In [ ]:
import json as _json
import mlflow.genai.datasets

UC_DATASET_TABLE = f"{CATALOG}.evaluations.refund_agent_eval_dataset"

# Read the dataset registered by stages/refund_setup.ipynb. The setup task
# JSON-stringifies any list-typed field on write (Arrow + mlflow.genai.datasets
# can't reliably serialize ARRAY<STRING>), so we deserialize symmetrically.
_rows = spark.table(UC_DATASET_TABLE).toPandas().to_dict(orient="records")


def _deserialize_record(rec):
    """Recursively undo the JSON-stringification done by setup."""
    out = {}
    for k, v in rec.items():
        if isinstance(v, dict):
            out[k] = _deserialize_record(v)
        elif isinstance(v, str):
            try:
                parsed = _json.loads(v)
                out[k] = parsed if isinstance(parsed, (list, dict)) else v
            except (ValueError, TypeError):
                out[k] = v
        else:
            out[k] = v
    return out


EVAL_DATASET = [_deserialize_record(r) for r in _rows]

if not EVAL_DATASET:
    raise RuntimeError(
        f"{UC_DATASET_TABLE} is empty. Re-run the Refund_Setup task first "
        "(it builds the dataset from {CATALOG}.lakeflow.all_events)."
    )

print(f"Loaded {len(EVAL_DATASET)} eval rows from {UC_DATASET_TABLE}")

import mlflow
import mlflow.genai

# Required: without databricks-uc the registry client hits workspace MLflow,
# where 3-part names are opaque strings and load_prompt raises NotFound.
mlflow.set_registry_uri("databricks-uc")

# Resolve the production prompt version so eval runs can be tagged with it.
# Lets a failing scorer point at exactly which prompt revision regressed.
PROMPT_NAME = "refund_system"
PROMPT_URI = f"prompts:/{CATALOG}.prompts.{PROMPT_NAME}@production"
try:
    PROMPT_VERSION = str(mlflow.genai.load_prompt(PROMPT_URI).version)
    print(f"  Resolved {PROMPT_URI} -> v{PROMPT_VERSION}")
except Exception as _exc:
    PROMPT_VERSION = "unknown"
    print(f"  Could not resolve {PROMPT_URI}: {type(_exc).__name__}: {_exc}")


In [ ]:
from mlflow.genai.scorers import Guidelines, Safety
try:
    from mlflow.genai.scorers import RelevanceToQuery
    _has_relevance = True
except ImportError:
    RelevanceToQuery = None
    _has_relevance = False
    print("⚠\ufe0f  RelevanceToQuery not available in this MLflow version — skipping it from the eval mix")

refund_reason = Guidelines(
    name="refund_reason",
    guidelines=[
        "If a refund is offered, its reason must relate to order timing, "
        "not to other issues such as missing components."
    ],
)

policy_alignment = Guidelines(
    name="policy_alignment",
    guidelines=[
        "If the response is a refund decision, it must explicitly state APPROVED or DENIED "
        "(or partial), and the decision must be consistent with the cited order/customer evidence. "
        "An approval without supporting evidence (or a denial that ignores documented order issues) fails."
    ],
)

cites_evidence = Guidelines(
    name="cites_evidence",
    guidelines=[
        "The response must cite specific evidence from the order or customer history \u2014 "
        "for example order ID, item names, prices, complaint history, refund amount, or timestamps. "
        "A response that just states a decision without grounding in the order data fails."
    ],
)

decision_clarity = Guidelines(
    name="decision_clarity",
    guidelines=[
        "The response must clearly explain WHY the refund decision was made, in 1\u20133 sentences, "
        "in language a customer support agent could read aloud. "
        "It must not be a JSON dump or unstructured reasoning trace."
    ],
)

_DATASET_SCORERS = [
    refund_reason,
    policy_alignment,
    cites_evidence,
    decision_clarity,
    Safety(),
]
if _has_relevance:
    _DATASET_SCORERS.append(RelevanceToQuery())

print(f"✅ Scorers defined ({len(_DATASET_SCORERS)}): " + ", ".join(s.name if hasattr(s, 'name') else type(s).__name__ for s in _DATASET_SCORERS))

In [ ]:
import mlflow
import mlflow.genai

_mlflow_dataset = mlflow.data.from_spark(
    spark.table(UC_DATASET_TABLE),
    table_name=UC_DATASET_TABLE,
)

with mlflow.start_run(run_name=f"{CATALOG}-refund-agent-eval"):
    mlflow.log_input(_mlflow_dataset, context="eval")
    mlflow.log_param("prompt_uri", PROMPT_URI)
    mlflow.log_param("prompt_version", PROMPT_VERSION)
    results = mlflow.genai.evaluate(
        data=EVAL_DATASET,
        scorers=_DATASET_SCORERS,
        predict_fn=predict_fn,
    )

print("\n✅ Evaluation complete")
print(f"  metrics: {getattr(results, 'metrics', 'N/A')}")

## Trace-derived eval — recent production calls

The block above runs a curated dataset against the deployed endpoint. That's
useful for catching regressions on **known-good scenarios**, but it doesn't
tell us how the agent is doing on **real production traffic**.

This section harvests the most recent successful traces from the prod
experiment (`/Shared/{CATALOG}_refund_agent_prod` — populated whenever the
streaming pipeline triggers a refund decision), turns them back into eval
inputs, and re-runs the agent against them with a global quality
guideline.

This mirrors the pattern in `stages/operational_evaluation.ipynb`
for the Operational Supervisor, so all three agents share the same flywheel:

```
Production traffic → MLflow traces → resampled eval dataset → quality scoring
```

Skipped gracefully if the prod experiment doesn't exist yet or has no
traces (fresh deploy with no traffic).

In [ ]:
import mlflow
import pandas as pd

prod_experiment_name = f"/Shared/{CATALOG}_refund_agent_prod"
trace_eval_data = []
traces_df = pd.DataFrame()

try:
    _exp = mlflow.get_experiment_by_name(prod_experiment_name)
except Exception as exc:
    print(f"  Could not look up prod experiment: {type(exc).__name__}: {exc}")
    _exp = None

if _exp is None:
    print(
        f"Skipping trace eval — experiment {prod_experiment_name} not found.\n"
        "  Send some refund decisions through the deployed endpoint first."
    )
else:
    print(f"Found prod traces experiment: {prod_experiment_name} ({_exp.experiment_id})")

    # `return_type="list"` is required — `mlflow.search_traces()` returns a
    # pandas DataFrame by default in MLflow 3, and iterating it yields column
    # names (strings).  The iteration below expects `Trace` objects with
    # `.data.spans` / `.info.request_id`, so we explicitly request the list form.
    traces = mlflow.search_traces(
        experiment_ids=[_exp.experiment_id],
        filter_string="status = 'OK'",
        max_results=50,
        order_by=["timestamp DESC"],
        return_type="list",
    )
    print(f"Pulled {len(traces)} OK-status traces (most recent 50).")

    records = []
    for trace in traces:
        try:
            root = trace.data.spans[0]
            inputs_raw = root.inputs or {}
            messages = inputs_raw.get("messages", [])
            question = next(
                (m["content"] for m in messages if m.get("role") == "user"),
                None,
            )
            if question:
                records.append({
                    "trace_id": trace.info.request_id,
                    "timestamp_ms": trace.info.timestamp_ms,
                    "latency_ms": trace.info.execution_time_ms,
                    "question": question,
                })
        except Exception as _te:
            print(f"  ⚠️  skipped trace {getattr(trace.info, 'request_id', '?')}: {_te}")
            continue

    traces_df = pd.DataFrame(records)
    print(f"Extracted {len(traces_df)} usable user-question records.")

    if not traces_df.empty:
        TRACE_EVAL_LIMIT = 20
        trace_eval_data = [
            {"inputs": {"messages": [{"role": "user", "content": row["question"]}]}}
            for _, row in traces_df.head(TRACE_EVAL_LIMIT).iterrows()
        ]
        print(f"Trace-derived eval dataset: {len(trace_eval_data)} records (cap={TRACE_EVAL_LIMIT}).")
        display(traces_df[["question", "latency_ms"]].head(10))

In [ ]:
from mlflow.genai.scorers import Guidelines, Safety

_TRACE_GUIDELINE = (
    "Response must include specific data points (numbers, locations, dates, "
    "or order identifiers). "
    "Response must not be a generic hedge or refusal. "
    "Response must directly address the refund question asked."
)

if trace_eval_data:
    _trace_df = pd.DataFrame(
        [{"query": row["inputs"]["messages"][0]["content"]} for row in trace_eval_data]
    )
    _trace_dataset = mlflow.data.from_pandas(
        _trace_df,
        source=f"{prod_experiment_name} (last {len(trace_eval_data)} OK traces)",
        name="refund_agent_trace_eval_dataset",
    )

    with mlflow.start_run(run_name=f"{CATALOG}-refund-agent-trace-eval"):
        mlflow.log_input(_trace_dataset, context="trace_eval")
        mlflow.log_param("prompt_uri", PROMPT_URI)
        mlflow.log_param("prompt_version", PROMPT_VERSION)
        trace_results = mlflow.genai.evaluate(
            data=trace_eval_data,
            scorers=[
                refund_reason,
                policy_alignment,
                cites_evidence,
                Guidelines(name="trace_quality", guidelines=_TRACE_GUIDELINE),
                Safety(),
                *([RelevanceToQuery()] if _has_relevance else []),
            ],
            predict_fn=predict_fn,
        )
    print("\n✅ Trace-derived evaluation complete")
    print(f"  metrics: {getattr(trace_results, 'metrics', 'N/A')}")
else:
    print("Skipping trace evaluation — no trace data available yet.")